In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar
import os
import glob

print("="*70)
print("McNemar's Test: Fine-Tuned XLM-RoBERTa vs Baselines")
print("="*70)

# Update these paths to your actual directories
FINETUNED_DIR = "xlmr_roberta_evaluation_results"  # Where fine-tuned model CSVs are
BASELINE_DIR = "zeroshot_baseline_results"  # Where zero-shot baseline CSVs are

def load_predictions_from_csv(filepath):
    """
    Load predictions from a CSV file.
    Returns dictionary with y_true, y_pred arrays (excluding parse failures)
    """
    try:
        df = pd.read_csv(filepath)
        
        # Filter out parse failures if they exist (prediction == 3)
        if 'prediction' in df.columns:
            valid_mask = df['prediction'] != 3
            df_filtered = df[valid_mask]
        else:
            df_filtered = df
        
        return {
            'y_true': df_filtered['Trigger'].values,
            'y_pred': df_filtered['prediction'].values,
            'correct': df_filtered['correct'].values,
            'filepath': filepath,
            'n_samples': len(df_filtered)
        }
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def find_csv_file(directory, model_name, language):
    """Find the most recent CSV file matching model and language"""
    pattern = os.path.join(directory, f"{model_name}*{language}*.csv")
    files = glob.glob(pattern)
    
    if not files:
        print(f"Warning: No files found matching {pattern}")
        return None
    
    # Get most recent file if multiple exist
    most_recent = max(files, key=os.path.getctime)
    return most_recent

def perform_mcnemar_test(model1_pred, model2_pred, y_true, model1_name, model2_name):
    """
    Perform McNemar's test comparing two models
    
    Returns:
        Dictionary with test results
    """
    # Ensure same samples
    assert len(model1_pred) == len(model2_pred) == len(y_true), \
        "Prediction arrays must have same length"
    
    # Calculate correctness
    correct1 = (model1_pred == y_true)
    correct2 = (model2_pred == y_true)
    
    # Create contingency table
    both_correct = np.sum(correct1 & correct2)
    only_1_correct = np.sum(correct1 & ~correct2)
    only_2_correct = np.sum(~correct1 & correct2)
    both_wrong = np.sum(~correct1 & ~correct2)
    
    # McNemar's test contingency table format
    contingency_table = np.array([[both_correct, only_1_correct],
                                   [only_2_correct, both_wrong]])
    
    # Perform test
    result = mcnemar(contingency_table, exact=False, correction=True)
    
    # Calculate accuracies
    acc1 = np.mean(correct1)
    acc2 = np.mean(correct2)
    
    return {
        'model1': model1_name,
        'model2': model2_name,
        'statistic': result.statistic,
        'p_value': result.pvalue,
        'both_correct': both_correct,
        'only_model1_correct': only_1_correct,
        'only_model2_correct': only_2_correct,
        'both_wrong': both_wrong,
        'model1_accuracy': acc1,
        'model2_accuracy': acc2,
        'significant': result.pvalue < 0.05,
        'n_samples': len(y_true)
    }


def print_test_result(result, language):
    """Pretty print McNemar's test results"""
    print(f"\n{'='*70}")
    print(f"Language: {language}")
    print(f"Comparing: {result['model1']} vs {result['model2']}")
    print(f"{'='*70}")
    print(f"Sample size: {result['n_samples']}")
    print(f"\nAccuracies:")
    print(f"  {result['model1']}: {result['model1_accuracy']:.4f}")
    print(f"  {result['model2']}: {result['model2_accuracy']:.4f}")
    print(f"  Difference: {result['model1_accuracy'] - result['model2_accuracy']:.4f}")
    
    print(f"\nContingency Table:")
    print(f"  Both correct:              {result['both_correct']}")
    print(f"  Only {result['model1'][:20]:20} correct: {result['only_model1_correct']}")
    print(f"  Only {result['model2'][:20]:20} correct: {result['only_model2_correct']}")
    print(f"  Both wrong:                {result['both_wrong']}")
    
    print(f"\nMcNemar's Test:")
    print(f"  Chi-squared statistic: {result['statistic']:.4f}")
    print(f"  P-value: {result['p_value']:.4f}")
    print(f"  Significance level: α = 0.05")
    
    if result['significant']:
        if result['only_model1_correct'] > result['only_model2_correct']:
            print(f"\nRESULT: {result['model1']} is SIGNIFICANTLY BETTER (p < 0.05)")
        else:
            print(f"\nRESULT: {result['model2']} is SIGNIFICANTLY BETTER (p < 0.05)")
    else:
        print(f"\nRESULT: No significant difference (p ≥ 0.05)")


# Define models to compare
FINETUNED_MODELS = [
    "XLM-R-Multilingual",
    "XLM-R-Arabic", 
    "XLM-R-English"
]

BASELINE_MODELS = [
    "XLM-R-Baseline-ZeroShot"
]

LANGUAGES = ["Arabic", "English"]

# Store all results
all_results = []

for language in LANGUAGES:
    print(f"\n{'#'*70}")
    print(f"# TESTING: {language.upper()}")
    print(f"{'#'*70}")
    
    # Load baseline predictions
    baseline_name = BASELINE_MODELS[0]
    baseline_file = find_csv_file(BASELINE_DIR, baseline_name, language)
    
    if baseline_file is None:
        print(f"Skipping {language}: Baseline file not found")
        continue
    
    baseline_data = load_predictions_from_csv(baseline_file)
    if baseline_data is None:
        continue
    
    print(f"\nLoaded baseline: {baseline_file}")
    print(f"Baseline samples: {baseline_data['n_samples']}")
    
    # Compare each fine-tuned model against baseline
    for ft_model in FINETUNED_MODELS:
        # Load fine-tuned model predictions
        ft_file = find_csv_file(FINETUNED_DIR, ft_model, language)
        
        if ft_file is None:
            print(f"Skipping {ft_model}: File not found")
            continue
        
        ft_data = load_predictions_from_csv(ft_file)
        if ft_data is None:
            continue
        
        print(f"\nLoaded fine-tuned: {ft_file}")
        print(f"Fine-tuned samples: {ft_data['n_samples']}")
        
        # Verify same test set
        if ft_data['n_samples'] != baseline_data['n_samples']:
            print(f"Warning: Sample size mismatch!")
            print(f"  Fine-tuned: {ft_data['n_samples']}, Baseline: {baseline_data['n_samples']}")
            # Take minimum to be safe
            min_samples = min(ft_data['n_samples'], baseline_data['n_samples'])
            ft_pred = ft_data['y_pred'][:min_samples]
            baseline_pred = baseline_data['y_pred'][:min_samples]
            y_true = ft_data['y_true'][:min_samples]
        else:
            ft_pred = ft_data['y_pred']
            baseline_pred = baseline_data['y_pred']
            y_true = ft_data['y_true']
        
        # Perform McNemar's test
        result = perform_mcnemar_test(
            ft_pred,
            baseline_pred,
            y_true,
            ft_model,
            baseline_name
        )
        
        result['language'] = language
        
        # Print results
        print_test_result(result, language)
        
        # Store for summary
        all_results.append({
            'Fine-tuned Model': ft_model,
            'Baseline': baseline_name,
            'Language': language,
            'FT Accuracy': result['model1_accuracy'],
            'Baseline Accuracy': result['model2_accuracy'],
            'Improvement': result['model1_accuracy'] - result['model2_accuracy'],
            'P-value': result['p_value'],
            'Significant (α=0.05)': 'Yes' if result['significant'] else '❌ No',
            'Chi-squared': result['statistic']
        })

# ========================================
# Summary Table
# ========================================
print("\n" + "="*70)
print("SUMMARY: Fine-tuned XLM-RoBERTa vs Zero-Shot Baseline")
print("="*70)

summary_df = pd.DataFrame(all_results)

if len(summary_df) > 0:
    print("\n" + summary_df.to_string(index=False))
    
    # Save results
    output_path = os.getcwd()
    
    # Save CSV
    csv_path = os.path.join(output_path, "mcnemar_test_results.csv")
    summary_df.to_csv(csv_path, index=False)
    print(f"\nResults saved to: {csv_path}")
    
    # Create LaTeX table
    latex_df = summary_df[['Fine-tuned Model', 'Language', 'FT Accuracy', 
                           'Baseline Accuracy', 'Improvement', 'P-value', 'Significant (α=0.05)']]
    
    latex_str = latex_df.to_latex(
        index=False,
        float_format="%.4f",
        caption="McNemar's Test: Fine-tuned XLM-RoBERTa vs Zero-Shot Baseline (α=0.05)",
        label="tab:mcnemar_finetuned_vs_baseline",
        escape=False
    )
    
    latex_path = os.path.join(output_path, "mcnemar_test_results.tex")
    with open(latex_path, 'w') as f:
        f.write(latex_str)
    print(f"LaTeX table saved to: {latex_path}")
    
    # ========================================
    # Key Findings
    # ========================================
    print("\n" + "="*70)
    print("KEY FINDINGS")
    print("="*70)
    
    # Count significant improvements
    sig_improvements = summary_df[summary_df['Significant (α=0.05)'] == 'Yes']
    print(f"\nSignificant improvements: {len(sig_improvements)} out of {len(summary_df)}")
    
    if len(sig_improvements) > 0:
        print("\nModels with significant improvement over baseline:")
        for _, row in sig_improvements.iterrows():
            print(f"  • {row['Fine-tuned Model']} ({row['Language']}): "
                  f"+{row['Improvement']:.4f} accuracy, p={row['P-value']:.4f}")
    
    # Best performing model
    best_model = summary_df.loc[summary_df['FT Accuracy'].idxmax()]
    print(f"\nBest performing fine-tuned model:")
    print(f"  {best_model['Fine-tuned Model']} ({best_model['Language']})")
    print(f"  Accuracy: {best_model['FT Accuracy']:.4f}")
    print(f"  Improvement over baseline: +{best_model['Improvement']:.4f}")
    
else:
    print("\nNo results to display. Check file paths.")

print("\n" + "="*70)
print("TESTING COMPLETE")
print("="*70)